**Program to create the distance data csv for image stitches C++ program**

- Input .txt file from the arduino serial output & image file name strings.
- Output CSV with the frame timestamps and corresponding distance measurements in mm.


In [26]:
import numpy as np
import pandas as pd
import re

In [27]:
# Path to the text file in Google Drive
file_path = '/content/drive/MyDrive/stitches/stitch2.txt'
images = [645, 4329, 5533, 9073]

data_records = []

try:
    with open(file_path, 'r') as f:
        for line in f:
            record = {}
            # Extract timestamp
            timestamp_match = re.match(r'(\d{2}:\d{2}:\d{2}\.\d{3})', line)
            if timestamp_match:
                record['Timestamp'] = timestamp_match.group(1)

            # Remove the timestamp and ' ->' part for easier parsing of key-value pairs
            processed_line = re.sub(r'\d{2}:\d{2}:\d{2}\.\d{3} -> ', '', line).strip()

            # Split by backslash to get individual measurements/readings
            measurements = processed_line.split('\\')

            for meas in measurements:
                # Extract key-value pairs like 'X: 9.17', 'pot: 37'
                kv_pairs = re.findall(r'([a-zA-Z]+):\s*([\d\.-]+)', meas)
                for key, value in kv_pairs:
                    record[key] = float(value) if '.' in value or '-' in value else int(value)

            if record: # Only add if a record was successfully parsed
                data_records.append(record)

    df = pd.DataFrame(data_records)

    # Rename 'pot' column to 'distance' if it exists
    if 'pot' in df.columns:
        df.rename(columns={'pot': 'distance'}, inplace=True)

    print(f"Successfully loaded and parsed data from '{file_path}'.")
    print("First 5 rows of the DataFrame:")
    display(df.head())

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure it exists and the path is correct.")
except Exception as e:
    print(f"An error occurred while reading or parsing the file: {e}")
    print("Please check the file format or provide more details if parsing errors persist.")

Successfully loaded and parsed data from '/content/drive/MyDrive/stitches/stitch2.txt'.
First 5 rows of the DataFrame:


,Timestamp,X,Y,Z,qW,qX,qY,qZ,distance
0,16:02:05.497,-40.19,-13.88,-28.00,0.6819,-0.2728,-0.6757,-0.0637,21
1,16:02:05.625,-39.75,-13.56,-27.19,0.6819,-0.2728,-0.6757,-0.0637,21
2,16:02:05.722,-40.19,-14.25,-26.75,0.6819,-0.2728,-0.6757,-0.0637,21
3,16:02:05.850,-39.00,-14.25,-27.19,0.6819,-0.2728,-0.6757,-0.0637,21
4,16:02:05.946,-40.19,-13.88,-27.19,0.6819,-0.2728,-0.6757,-0.0637,21


In [28]:
# Ensure 'Timestamp' column is in datetime format for calculations
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%H:%M:%S.%f')

# Get the first timestamp
first_timestamp = df['Timestamp'].iloc[0]

# Calculate the difference between each timestamp and the first timestamp in milliseconds
df['timestamp_difference_ms'] = (df['Timestamp'] - first_timestamp).dt.total_seconds() * 1000

# Calculate distance in mm
df['distance_mm'] = df['distance'] * 750 / 1025

# Create the new DataFrame with the required columns
stitches_output_dt = df[['timestamp_difference_ms', 'distance_mm']].copy()

# The first entry of 'timestamp_difference_ms' will automatically be 0 with this calculation

print("Recalculated 'timestamp_difference_ms' relative to the first timestamp.")
print("First 5 rows of the updated 'stitches_output_dt' DataFrame:")
display(stitches_output_dt.head(200))

Recalculated 'timestamp_difference_ms' relative to the first timestamp.
First 5 rows of the updated 'stitches_output_dt' DataFrame:


,timestamp_difference_ms,distance_mm
0,0.0,15.365854
1,128.0,15.365854
2,225.0,15.365854
3,353.0,15.365854
4,449.0,15.365854
...,...,...
195,22015.0,151.463415
196,22145.0,151.463415
197,22240.0,151.463415
198,22369.0,152.195122


In [29]:
matched_data = []

# Convert timestamp_difference_ms to a numpy array for efficient comparison
time_diff_array = stitches_output_dt['timestamp_difference_ms'].values

for image_num in images:
    # Calculate the absolute difference between the image_num and all timestamps
    closest_idx = np.argmin(np.abs(time_diff_array - image_num))

    # Get the closest timestamp and its corresponding distance
    closest_timestamp = stitches_output_dt.loc[closest_idx, 'timestamp_difference_ms']
    corresponding_distance = stitches_output_dt.loc[closest_idx, 'distance_mm']

    matched_data.append({
        'Image Timestamp (ms)': closest_timestamp,
        'Distance (mm)': corresponding_distance
    })

# Create the new DataFrame
image_distance_df = pd.DataFrame(matched_data)

print("Created 'image_distance_df' with matched image timestamps and distances:")
display(image_distance_df)

Created 'image_distance_df' with matched image timestamps and distances:


,Image Timestamp (ms),Distance (mm)
0,675.0,15.365854
1,4313.0,30.000000
2,5535.0,38.048780
3,9043.0,73.902439


In [32]:
# Define the output CSV file path
output_csv_path = 'stitches2.csv'

# Export the DataFrame to a CSV file, without writing the index
image_distance_df.to_csv(output_csv_path, index=False)

print(f"DataFrame successfully exported to '{output_csv_path}'")

DataFrame successfully exported to 'stitches2.csv'


In [34]:
# Define the output text file path
output_txt_path = 'stitches2.txt'

# Export the DataFrame to a text file with space separation, no index, and no header
image_distance_df.to_csv(output_txt_path, sep=' ', index=False, header=False)

print(f"DataFrame successfully exported to '{output_txt_path}' as a space-separated text file.")

DataFrame successfully exported to 'stitches2.txt' as a space-separated text file.
